In [18]:
"""
Process PacBio CCS reads into barcode / substrate / kinase lookup tables.

Refactored from the original notebook-style code to a reusable function
and two example runs (blib1 and blib5).
"""
# pip install python-Levenshtein tqdm

import os
import gzip

import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq
from tqdm import tqdm

import os
from typing import List, Tuple
import numpy as np
import pandas as pd
from tqdm import tqdm

## process pacbio generated lookup table for blib screening

In [19]:
def make_pacbio_lookup(
    datadir: str,
    filename: str,
    out_path: str,
    rc_trigger: str = "CGTACGCACAATTC",
    barcode_left: str = "TATCCGGCTCGTATGTTGTGTGGAATTGTGCGTACG",
    barcode_right: str = "CCTAGGCAGCTATGACCATGATT",
    ks_left: str = "GGCGGTGGCGGGTCTGGTGGAGGAGGGTCAGGTGGTGGCGGCTCGGGAGGTGGAGCTAGC",
    ks_right: str = "AAGCTTATTAGTTATGTCAC",
    linker: str = "GGTGGATCAGGTGGTGGTGGTTCTGGTGGC",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Build a lookup table from a PacBio .fasta.gz file.

    Parameters
    ----------
    datadir : str
        Directory containing the FASTA file.
    filename : str
        Name of the .fasta.gz file.
    out_path : str
        Output path for the lookup table (.txt / .tsv).
    rc_trigger : str
        Sequence used to decide whether to reverse-complement the read.
    barcode_left, barcode_right : str
        Flanking sequences used to extract the barcode.
    ks_left, ks_right : str
        Flanking sequences used to extract the ksregion.
    linker : str
        Sequence separating substrate and kinase within ksregion.
    verbose : bool
        If True, print shapes at each filtering step.

    Returns
    -------
    pd.DataFrame
        Filtered dataframe with columns:
        [identifier, length, seq, barcode_seq, ksregion, substrate, kinase]
    """
    input_path = os.path.join(datadir, filename)
    if verbose:
        print(f"Reading {input_path}")

    # --- Read sequences ---
    identifiers = []
    lengths = []
    seqs = []

    with gzip.open(input_path, "rt") as handle:
        for seq_record in tqdm(SeqIO.parse(handle, "fasta")):
            identifiers.append(seq_record.id)
            lengths.append(len(seq_record.seq))
            seqs.append(str(seq_record.seq))

    # --- Reverse complement where needed ---
    processed_seq = []
    for s in seqs:
        if rc_trigger in s:
            processed_seq.append(str(Seq(s).reverse_complement()))
        else:
            processed_seq.append(s)

    dat = pd.DataFrame(
        {
            "identifier": identifiers,
            "length": lengths,
            "seq": processed_seq,
        }
    )

    if verbose:
        print("Initial dat shape:", dat.shape)

    # --- Filter for all required pieces of the construct ---
    dat_filtered = dat.loc[dat["seq"].apply(lambda s: barcode_left in s), :]
    if verbose:
        print("After barcode_left filter:", dat_filtered.shape)

    dat_filtered = dat_filtered.loc[
        dat_filtered["seq"].apply(lambda s: barcode_right in s), :
    ]
    if verbose:
        print("After barcode_right filter:", dat_filtered.shape)

    dat_filtered = dat_filtered.loc[
        dat_filtered["seq"].apply(lambda s: ks_left in s), :
    ]
    if verbose:
        print("After ks_left filter:", dat_filtered.shape)

    dat_filtered = dat_filtered.loc[
        dat_filtered["seq"].apply(lambda s: ks_right in s), :
    ]
    if verbose:
        print("After ks_right filter:", dat_filtered.shape)

    # --- Extract barcode and ksregion ---
    dat_filtered["barcode_seq"] = dat_filtered["seq"].apply(
        lambda s: s.split(barcode_left)[1].split(barcode_right)[0]
    )

    dat_filtered["ksregion"] = dat_filtered["seq"].apply(
        lambda s: s.split(ks_left)[1].split(ks_right)[0]
    )

    # --- Keep only reads with the linker and split into substrate / kinase ---
    dat_filtered = dat_filtered.loc[
        dat_filtered["ksregion"].apply(lambda s: linker in s), :
    ]
    if verbose:
        print("After linker filter:", dat_filtered.shape)

    dat_filtered["substrate"] = dat_filtered["ksregion"].apply(
        lambda s: s.split(linker)[0]
    )
    dat_filtered["kinase"] = dat_filtered["ksregion"].apply(
        lambda s: s.split(linker)[1]
    )

    # --- Write out ---
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    dat_filtered.to_pickle(out_path)
    if verbose:
        print(f"Wrote lookup table to: {out_path}")

    return dat_filtered

In [24]:
OUTDIR = "./lookup_tables"

os.makedirs(OUTDIR, exist_ok=True)

# ---- Run 1: blib1 (labeled reversed so Y )----
blib1_datadir = "${KSI_DATA_ROOT}/20231106_Yu_C/blib1-ksb.CY/CCS_12462/"
blib1_filename = "m64076_231104_004250.hifi_reads.fasta.gz"
blib1_out = os.path.join(OUTDIR, "y_pacbio_lookup_raw.pkl")
y_lookup = make_pacbio_lookup(
    datadir=blib1_datadir,
    filename=blib1_filename,
    out_path=blib1_out,
    verbose=True,
)


# ---- Run 2: blib5 (labeled reversed so ST )----
blib5_datadir = "${KSI_DATA_ROOT}/20231106_Yu_C/blib5-ksb.CY/CCS_12461/"
blib5_filename = "m64076_231104_154224.hifi_reads.fasta.gz"
blib5_out = os.path.join(OUTDIR, "st_pacbio_lookup_raw.pkl")
st_lookup = make_pacbio_lookup(
    datadir=blib5_datadir,
    filename=blib5_filename,
    out_path=blib5_out,
    verbose=True,
)

Reading ${KSI_DATA_ROOT}/20231106_Yu_C/blib1-ksb.CY/CCS_12462/m64076_231104_004250.hifi_reads.fasta.gz


2381935it [00:38, 61682.57it/s]


Initial dat shape: (2381935, 3)
After barcode_left filter: (1973956, 3)
After barcode_right filter: (1935748, 3)
After ks_left filter: (1800470, 3)
After ks_right filter: (1755642, 3)
After linker filter: (1691475, 5)
Wrote lookup table to: ./lookup_tables/y_pacbio_lookup_raw.pkl
Reading ${KSI_DATA_ROOT}/20231106_Yu_C/blib5-ksb.CY/CCS_12461/m64076_231104_154224.hifi_reads.fasta.gz


3039052it [00:48, 62249.99it/s]


Initial dat shape: (3039052, 3)
After barcode_left filter: (2495983, 3)
After barcode_right filter: (2445887, 3)
After ks_left filter: (2268964, 3)
After ks_right filter: (2215632, 3)
After linker filter: (2133880, 5)
Wrote lookup table to: ./lookup_tables/st_pacbio_lookup_raw.pkl


## Process PacBio barcode→kinase lookup tables:
1. dist_corrected == 0
2. barcode length == 19
3. one barcode correponding to multiple ks pairs needs to be filtered

In [25]:
blib_kinases_with_acc = pd.read_csv("../screening_analysis/blib_kinases_with_acc.csv",header = None)
blib_kinases_with_acc.columns = ['kinase_type','kinase_name','kinase_aa','kinase_acc']
# 251210: fixing the missing DYRK4 issue by putting manually fixing the incorrect kinase sequence
blib_kinases_with_acc.loc[blib_kinases_with_acc['kinase_name']=='DYRK4_HUMAN','kinase_aa']='YEVLETIGKGSFGQVAKCLDHKNNELVALKIIRNKKRFHQQALMELKILEALRKKDKDNTYNVVHMKDFFYFRNHFCITFELLGINLYELMKNNNFQGFSLSIVRRFTLSVLKCLQMLSVEKIIHCDLKPENIVLYQKGQASVKVIDFGSSCYEHQKVYTYIQSRFYRSPEVILGHPYDVAIDMWSLGCITAELYTGYPLFPGENEVEQLACIMEVLGLPPAGFIQTASRRQTFFDSKGFPKNITNNRGKKRYPDSKDLTMVLKTYDTSFLDFLRRCLVWEPSLRMTPDQALKHAWI'
blib_kinases_with_acc.to_csv('./lookup_tables/blib_kinases_with_acc.csv')

In [26]:
def build_canonical_lookup(
    df,
    blib_kinases_with_acc,
    corrected_out: str,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    """
    Load a lookup table, infer canonical kinase sequences, correct all reads,
    and sanitize to one best read per barcode.

    Parameters
    ----------
   
    corrected_out : str
        Output path (pickle) for the corrected lookup table.
    sanitized_out : str
        Output path (pickle) for the sanitized lookup table
        (one best row per barcode).
    min_barcode_len : int, default 15
        Minimum barcode length to retain a read.
    verbose : bool, default True
        Print progress and small summaries.

    Returns
    -------
    corrected_df : pd.DataFrame
        Lookup table with columns `kinase_corrected` and `dist_corrected`.
    sanitized_df : pd.DataFrame
        Deduplicated table: one best row (min dist_corrected) per barcode.
    canonical_forms : list of str
        Canonical kinase sequences used.
    """    
    if verbose:
        print("Initial shape:", df.shape)    
    
    df = df.loc[df['barcode_seq'].apply(lambda s: len(s)==19),:]

    df['substrate_aa'] = df['substrate'].apply(lambda s: str(Seq(s).translate()))
    df = df.loc[df["substrate_aa"].apply(lambda s:len(s)==15),:]
    df = df.loc[~df["substrate_aa"].apply(lambda s: "*" in s),:]
    
    df['kinase_aa'] = df['kinase'].apply(lambda s: str(Seq(s).translate()))
    df = df.loc[df["kinase_aa"].apply(lambda s: s[-1]=='*'),:]
    df['kinase_aa'] = df['kinase_aa'].apply(lambda s: s[:-1])
    df = df.loc[~df["kinase_aa"].apply(lambda s: "*" in s),:]

    if verbose:
        print("after filtering on barcode length and no nonsense mut and stop after kinase:", df.shape)
        print("# unique barcodes:", df["barcode_seq"].nunique())
        print(
            "Reads per unique barcode (mean):",
            df.shape[0] / df["barcode_seq"].nunique(),
        )
        
    # Count reads per (barcode, kinase, substrate)
    pair_counts = (
        df.groupby(["barcode_seq", "kinase_aa", "substrate_aa"], dropna=False)
          .size()
          .rename("n_reads")
          .reset_index()
    )
    
    # Fraction within each barcode
    pair_counts["total_reads"] = pair_counts.groupby("barcode_seq")["n_reads"].transform("sum")
    pair_counts["frac"] = pair_counts["n_reads"] / pair_counts["total_reads"]
    
    # Dominant assignment per barcode
    dominant = pair_counts.loc[pair_counts.groupby("barcode_seq")["frac"].idxmax()].copy()
    
    # Keep only barcodes whose dominant assignment is >= 90%
    keep = dominant.loc[dominant["frac"] >= 0.90, ["barcode_seq", "kinase_aa", "substrate_aa"]]
    
    # Filter df down to only the dominant assignment rows for kept barcodes
    df = df.merge(keep, on=["barcode_seq", "kinase_aa", "substrate_aa"], how="inner")
    
    # (optional) collapse to one row per barcode/pair if that's what you want downstream
    df = df.drop_duplicates(["barcode_seq", "kinase_aa", "substrate_aa"])
    
    # Continue as before
    df = df.merge(blib_kinases_with_acc, on="kinase_aa", how="left")

    
    if verbose:
        print("after filtering on bijection and kinase in the screening pool:", df.shape)
        print("# unique barcodes:", df["barcode_seq"].nunique())
        print(
            "Reads per unique barcode (mean):",
            df.shape[0] / df["barcode_seq"].nunique(),
        )
        
    # Save corrected table
    os.makedirs(os.path.dirname(corrected_out), exist_ok=True)
    df.to_pickle(corrected_out)
    if verbose:
        print(f"Saved corrected lookup to: {corrected_out}")

    return df

In [27]:
LOOKUP_DIR = "./lookup_tables/"
y_lookup = pd.read_pickle('./lookup_tables/y_pacbio_lookup_raw.pkl')
st_lookup = pd.read_pickle('./lookup_tables/st_pacbio_lookup_raw.pkl')


# --- Run 1: blib5 → st_lookup ---
st_corrected_out = os.path.join(LOOKUP_DIR, "st.lookup.corrected.pkl")

st_corrected = build_canonical_lookup(
    df = st_lookup,
    blib_kinases_with_acc = blib_kinases_with_acc,
    corrected_out=st_corrected_out,
    verbose=True,
)

# --- Run 2: blib1 → y_lookup ---
y_corrected_out = os.path.join(LOOKUP_DIR, "y.lookup.corrected.pkl")

y_corrected = build_canonical_lookup(
    df=y_lookup,
    blib_kinases_with_acc = blib_kinases_with_acc,
    corrected_out=y_corrected_out,
    verbose=True,
)
print('251210')

Initial shape: (2133880, 7)


/opt/conda/envs/pytorch_env/lib/python3.11/site-packages/Bio/Seq.py:2877: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/tmp/ipykernel_2359/182827160.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['substrate_aa'] = df['substrate'].apply(lambda s: str(Seq(s).translate()))
/opt/conda/envs/pytorch_env/lib/python3.11/site-packages/Bio/Seq.py:2877: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


after filtering on barcode length and no nonsense mut and stop after kinase: (1631649, 9)
# unique barcodes: 144716
Reads per unique barcode (mean): 11.27483484894552
after filtering on bijection and kinase in the screening pool: (121070, 12)
# unique barcodes: 121070
Reads per unique barcode (mean): 1.0
Saved corrected lookup to: ./lookup_tables/st.lookup.corrected.pkl
Initial shape: (1691475, 7)


/opt/conda/envs/pytorch_env/lib/python3.11/site-packages/Bio/Seq.py:2877: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/tmp/ipykernel_2359/182827160.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['substrate_aa'] = df['substrate'].apply(lambda s: str(Seq(s).translate()))
/opt/conda/envs/pytorch_env/lib/python3.11/site-packages/Bio/Seq.py:2877: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


after filtering on barcode length and no nonsense mut and stop after kinase: (1286517, 9)
# unique barcodes: 242424
Reads per unique barcode (mean): 5.306887931887932
after filtering on bijection and kinase in the screening pool: (205272, 12)
# unique barcodes: 205272
Reads per unique barcode (mean): 1.0
Saved corrected lookup to: ./lookup_tables/y.lookup.corrected.pkl
251210
